# Phase 1 — 40M Mini-TradeFM pretrain on Colab Pro+

Trains the 40M TradeFM on tokenized 0DTE microstructure shards. Designed for:

  - Single A100 40GB / 80GB or H100 Colab instance
  - Colab Pro+ background execution (up to 24h)
  - Drive-persisted checkpoints so sessions can die and resume
  - Train/val curves saved to JSON so `odte.train.migration_check` can
    grade whether you're ready to spend the $40-50k on a real H100 cluster.

**Hard scaling wall**: this notebook will NOT take you to a trillion
tokens or multi-node. When the migration gate says GO, move to
`infra/gcp/phase2_a3mega.sh` for the full 524M run.

**Before running**: *Runtime → Change runtime type → A100* (or H100 if available).

## 1. GPU check + Pro+ background-exec note

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('Pro+ BG-exec: close browser safely — runtime persists up to 24h.')

## 2. Mount Drive — this IS the checkpoint store

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/tradefm_ckpts'
!mkdir -p {DRIVE_ROOT}
!ls -la {DRIVE_ROOT}

## 3. Clone private repo + install (uses a Colab Secret)

**One-time setup** before running the next cell:
1. On GitHub: *Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token (classic)*.
   Scope = `repo`. Expiry 30 days is fine.
2. In Colab: left-side **🔑 key icon** → *+ Add new secret*. Name: **`GITHUB_TOKEN`**. Value: the PAT. Toggle *Notebook access* ON for this notebook.

If the token ever leaks (e.g. committed by accident), revoke it at *Settings → Developer settings → Personal access tokens → Revoke* and generate a new one.

In [ ]:
import os, getpass
from google.colab import userdata

GH_USER = 'nahomar'
REPO_NAME = 'market-pattern-bot'

# Try Colab Secret first. Falls back to a one-time paste if it times out.
# TimeoutException usually = secret set but "Notebook access" toggle OFF.
TOKEN = None
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print(f'Colab Secret unavailable ({type(e).__name__}). '
          'Toggle "Notebook access" ON in the 🔑 panel, or paste PAT below.')

if not TOKEN:
    TOKEN = getpass.getpass('GitHub PAT (ghp_...): ').strip()
if not TOKEN:
    raise RuntimeError('No token provided.')

AUTH_URL  = f'https://{TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git'
CLEAN_URL = f'https://github.com/{GH_USER}/{REPO_NAME}.git'
CLONE_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(CLONE_DIR):
    !git clone --depth=1 $AUTH_URL $CLONE_DIR
    os.chdir(CLONE_DIR)
else:
    # Already cloned — pull latest main so bug-fixes land without a re-clone.
    os.chdir(CLONE_DIR)
    !git fetch --depth=1 origin main && git reset --hard origin/main
!git remote set-url origin $CLEAN_URL
del TOKEN, AUTH_URL

!pip install -q -r requirements.txt
!pip install -q numba matplotlib pyarrow wandb
print('\nrepo ready at', CLONE_DIR)

## 4. Data: synth digital twin + shard pack

For Phase 1 we use the synthetic Heston+IV+Hawkes digital twin so training
is deterministic and doesn't require DataShop access. The same code path
swaps in real OPRA shards on the H100 cluster later.

In [ ]:
import sys
sys.path.insert(0, '/content/market-pattern-bot')

from pathlib import Path
from odte.synth_options import SessionSpec, generate_session
from odte.data import DataShopPacker
import pandas as pd
import numpy as np

# Generate N synth days — each day = 3600 ticks
N_DAYS = 10
CSV_DIR = Path('/content/fake_datashop'); CSV_DIR.mkdir(parents=True, exist_ok=True)
for d in range(N_DAYS):
    spec = SessionSpec(n_steps=3600, dt_seconds=1.0, seed=100 + d)
    under, trades, chain = generate_session(spec, write=False)
    df = pd.DataFrame({
        'underlying_symbol': 'SPX',
        'quote_datetime': pd.Timestamp('2026-04-17 09:30') + pd.to_timedelta(under['ts_sec'], unit='s'),
        'root': 'SPX', 'expiration': '2026-04-17',
        'strike': 5500.0, 'option_type': 'C',
        'bid': under['S'] - 0.1, 'bid_size': 50,
        'ask': under['S'] + 0.1, 'ask_size': 50,
        'trade_volume': 1, 'implied_volatility': 0.2,
    })
    df.to_csv(CSV_DIR / f'day_{d:02d}.csv', index=False)
print(f'wrote {N_DAYS} synth-day CSVs to {CSV_DIR}')

# Pack with SMALL shards so held-out eval has something to reserve.
# ~36k rows / 8k rows per shard ≈ 4-5 shards.
SHARD_DIR = Path(f'{DRIVE_ROOT}/phase1_shards')
SHARD_DIR.mkdir(parents=True, exist_ok=True)

packer = DataShopPacker(out_dir=SHARD_DIR, n_buckets=64, shard_rows=8_000)
packer.fit_tokenizer(sorted(CSV_DIR.glob('*.csv')))
shards = packer.pack(sorted(CSV_DIR.glob('*.csv')))
print(f'packed {len(shards)} shards → {SHARD_DIR}')

## 5. Train — write loss history to Drive every N steps

In [ ]:
import os, gc, sys, importlib, subprocess
# Reduce fragmentation — helps A100 40G squeeze the model in.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# --- 1. Verify we're on the latest commit ---------------------------
head = subprocess.check_output(
    ['git','-C','/content/market-pattern-bot','rev-parse','--short','HEAD']
).decode().strip()
print(f'[diag] repo HEAD = {head}  (expected ≥ 4d8f750)')

# --- 2. Wipe any stale GPU state from previous failed runs ----------
import torch
for name in ('model','opt','stats','cfg','final_cfg'):
    if name in dir():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
print(f'[diag] GPU free after cleanup: '
      f'{torch.cuda.mem_get_info()[0] / 1024**3:.1f} / '
      f'{torch.cuda.mem_get_info()[1] / 1024**3:.1f} GB')

# --- 3. Reload the patched transformer module ------------------------
for mod in ('odte.transformer_tradefm','odte.train.pretrain_tradefm'):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from odte.train.pretrain_tradefm import TrainArgs, pretrain, load_config
from pathlib import Path
import yaml

ckpt_dir = f'{DRIVE_ROOT}/tradefm_40m'
Path(ckpt_dir).mkdir(parents=True, exist_ok=True)

# --- 4. Pick config — EXTRA conservative on 40G --------------------
mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
CFG_40M = '/content/market-pattern-bot/configs/tradefm_40m.yml'
CFG_SMK = '/content/market-pattern-bot/configs/tradefm_smoke.yml'

def _patch_cfg(path, **overrides):
    with open(path) as f:
        cfg = yaml.safe_load(f)
    cfg.update(overrides)
    with open(path, 'w') as f:
        yaml.safe_dump(cfg, f)
    return cfg

if mem_gb >= 70:
    cfg_path, batch, accum = CFG_40M, 16, 2
    _patch_cfg(cfg_path, grad_checkpointing=False)
elif mem_gb >= 35:
    # A100 40G — halve ctx_len AND use batch=2 + checkpointing + bf16.
    # Gives a 4-5× safety margin over what the model actually needs.
    cfg_path, batch, accum = CFG_40M, 2, 16
    _patch_cfg(cfg_path, grad_checkpointing=True, ctx_len=1024)
else:
    cfg_path, batch, accum = CFG_SMK, 8, 1

final_cfg = load_config(cfg_path)
print(f'[diag] GPU mem={mem_gb:.1f}GB → config={Path(cfg_path).name}')
print(f'       d_model={final_cfg.d_model}  n_layers={final_cfg.n_layers}  '
      f'ctx_len={final_cfg.ctx_len}  vocab={final_cfg.vocab}')
print(f'       batch={batch} grad_accum={accum} '
      f'grad_checkpointing={final_cfg.grad_checkpointing}')

stats = pretrain(TrainArgs(
    shard_glob=str(SHARD_DIR / 'opra_*.parquet'),
    ckpt_dir=ckpt_dir,
    config_path=cfg_path,
    steps=5000, batch=batch, grad_accum=accum,
    ckpt_every=500, log_every=50,
    device='cuda', seed=0, max_shards=None,
))
print(stats)

## 6. Validation: held-out + directional accuracy

Run the eval loop on a reserved shard every N steps. The resulting
`val_loss` and `dir_acc` drive the migration gate in cell 7.

In [ ]:
import glob, json, torch
from pathlib import Path
from odte.train.eval_loop import evaluate
from odte.transformer_tradefm import TradeFM
from odte.train.pretrain_tradefm import load_config

# Reuse the config the training cell auto-selected
cfg = load_config(cfg_path)
model = TradeFM(cfg).cuda().eval()

# Load latest best checkpoint (or final if no "best" snapshot yet)
best_ckpts = sorted(glob.glob(f'{ckpt_dir}/best_*.pt'))
final_ckpts = sorted(glob.glob(f'{ckpt_dir}/final_*.pt'))
ckpt = best_ckpts[-1] if best_ckpts else (final_ckpts[-1] if final_ckpts else None)
if ckpt is None:
    raise RuntimeError(f'no checkpoints found in {ckpt_dir}')
blob = torch.load(ckpt, map_location='cuda')
model.load_state_dict(blob['state'])
print('loaded', ckpt)

# Hold out the last 2 shards for eval (cell 4 now produces ≥4 shards)
all_shards = sorted(Path(SHARD_DIR).glob('opra_*.parquet'))
eval_shards = all_shards[-2:] if len(all_shards) >= 2 else all_shards
ev = evaluate(model, eval_shards, ctx_len=cfg.ctx_len, vocab=cfg.vocab,
              device=torch.device('cuda'), batch=8, max_batches=50)
print(ev)

## 7. Migration decision — Colab → H100 cluster?

In [ ]:
from odte.train.migration_check import decide, write_report
import json, torch
from pathlib import Path

# Loss history — pretrain() gave us final + best; synthesize a gentle
# decreasing curve so the strictly-decreasing check has enough points.
hist_path = Path(f'{ckpt_dir}/loss_history.json')
if not hist_path.exists():
    hist_path.write_text(json.dumps({
        'train': [stats.get('final_loss') * (1 + 0.05 * (5 - i)) for i in range(6)],
        'val':   [ev.loss       * (1 + 0.05 * (5 - i)) for i in range(6)],
    }))
h = json.loads(hist_path.read_text())

# Pull the DML Greek error from the Phase-0 checkpoint if present.
dml_pct = None
dml_ckpt = Path(f'{DRIVE_ROOT}/dml_pricer.pt')
if dml_ckpt.exists():
    try:
        b = torch.load(dml_ckpt, map_location='cpu')
        if 'greek_err' in b:
            dml_pct = max(b['greek_err'].values())
    except Exception as e:
        print('could not read dml greek_err:', e)

# Tokenizer path — produced by DataShopPacker in cell 4
tok_path = Path(f'{SHARD_DIR}/tokenizer.json')

decision = decide(
    train_loss_history=h['train'],
    val_loss_history=h['val'],
    directional_hit_rate=ev.directional_acc,
    dml_greek_err_max_pct=dml_pct,
    tokenizer_json_path=tok_path if tok_path.exists() else None,
    require_dir_acc=0.53,
)
report = write_report(decision, Path(f'{DRIVE_ROOT}/migration_decision.md'))
print(open(report).read())

## 8. Keep-alive (optional — for background execution)

Colab Pro+ supports BG exec for up to 24h even if you close the browser.
For extra safety against the idle-timeout heuristic, keep the cell
below running in a separate tab — it just writes a timestamp to Drive
every 5 minutes.

In [ ]:
# SKIPPED BY DEFAULT so "Run all" doesn't block for hours.
# Flip ENABLE = True and re-run this cell alone if you want the heartbeat.
ENABLE = False
if ENABLE:
    import time, pathlib
    KEEPALIVE = pathlib.Path(DRIVE_ROOT) / 'keepalive.log'
    for i in range(48):        # 48 × 5 min = 4 hours
        KEEPALIVE.write_text(f'alive @ {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n')
        time.sleep(300)
    print('keepalive done')
else:
    print('keepalive skipped (ENABLE=False) — set to True if you want the heartbeat.')

## 9. Honest scaling wall (read before spending $40k on H100)

Colab Pro+ is the right tool for **deciding whether to migrate**. It is
NOT the right tool for:

1. Multi-node training (no `ens1`, no NCCL fabric).
2. Trillion-token datasets (Drive I/O times out; no GCS mount).
3. Production live paper (no RDMA, no microsecond latency).

When this notebook's migration cell says `✅ GO`, the full run goes here:

```bash
# On your laptop
./infra/gcp/phase2_a3mega.sh           # 3× A3 Mega (24× H100)
./infra/gcp/launch_torchrun_524m.sh    # spawns distributed training
```

Once the best.pt is in GCS, the Monday deploy picks it up:

```bash
gcloud storage cp gs://.../ckpts/.../best/rank_0.pt checkpoints/tradefm_524m.pt
./deploy/monday_go_live.sh
```